# 4. KNN Clustering — จัดกลุ่มหน่วยเลือกตั้งตามพฤติกรรม
ใช้ K-Means แบ่งหน่วยเป็นกลุ่ม เช่น ฐานเสียงแน่น / swing zone / low engagement

**Features ที่ใช้:** สัดส่วนคะแนนพรรคหลัก + turnout + invalid rate

In [1]:
import sys
!{sys.executable} -m pip install folium scikit-learn matplotlib seaborn -q

In [2]:
import pandas as pd
import numpy as np
import folium
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings; warnings.filterwarnings('ignore')

CLEAN = Path('../cleaned/')
results = pd.read_csv(CLEAN / 'master_results_cleaned.csv')
summary = pd.read_csv(CLEAN / 'master_summary_cleaned.csv')
coords  = pd.read_csv(CLEAN / 'coords_cleaned.csv')

JOIN = ['district', 'sub-district', 'unit_number']

# --- สัดส่วนคะแนน บช รายหน่วย (top 5 พรรค) ---
bch = results[(results['unit_number'] != -1) & (results['type'] == 'บช')]
top_parties = ['ประชาชน', 'เพื่อไทย', 'กล้าธรรม', 'ภูมิใจไทย', 'รวมไทยสร้างชาติ']

unit_total = bch.groupby(JOIN)['score'].sum().reset_index().rename(columns={'score': 'total_bch'})
party_wide = bch[bch['name'].isin(top_parties)].pivot_table(
    index=JOIN, columns='name', values='score', aggfunc='sum', fill_value=0
).reset_index()
party_wide = party_wide.merge(unit_total, on=JOIN)
for p in top_parties:
    if p in party_wide.columns:
        party_wide[f'share_{p}'] = party_wide[p] / party_wide['total_bch']

# --- turnout + invalid rate รายหน่วย ---
s = summary[summary['unit_number'] != -1].copy()
s['turnout']      = s['valid_ballots']   / s['total_ballots']
s['invalid_rate'] = s['invalid_ballots'] / s['total_ballots']

# --- merge ทุกอย่าง ---
feat_cols = [f'share_{p}' for p in top_parties if f'share_{p}' in party_wide.columns]
df = party_wide[JOIN + feat_cols].merge(s[JOIN + ['turnout','invalid_rate']], on=JOIN, how='inner')
df = df.dropna()

# --- K-Means (k=4) ---
X = df[feat_cols + ['turnout', 'invalid_rate']].values
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# ตั้งชื่อ cluster จาก centroid
centers = pd.DataFrame(kmeans.cluster_centers_, columns=feat_cols + ['turnout', 'invalid_rate'])
cluster_labels = {}
for c in range(4):
    row = centers.iloc[c]
    dominant = row[feat_cols].idxmax().replace('share_', '')
    to = row['turnout']
    if to > 0.75:
        label = f'ฐานเสียง {dominant}'
    elif to < 0.60:
        label = f'Low Engagement ({dominant})'
    else:
        label = f'Swing ({dominant})'
    cluster_labels[c] = f'C{c}: {label}'

df['label'] = df['cluster'].map(cluster_labels)
print('จำนวนหน่วยต่อ cluster:')
print(df['label'].value_counts().to_string())
print()
print('ค่าเฉลี่ยรายกลุ่ม:')
print(df.groupby('label')[feat_cols + ['turnout','invalid_rate']].mean().round(3).to_string())

# --- Plot บนแผนที่ ---
df_map = df.merge(coords, on=JOIN, how='inner')
agg_map = df_map.groupby(['district','latitude','longitude','label']).size().reset_index(name='count')

palette = ['#e74c3c','#3498db','#2ecc71','#9b59b6']
label_list = sorted(df['label'].unique())
color_map = {l: palette[i] for i, l in enumerate(label_list)}

center = [df_map['latitude'].mean(), df_map['longitude'].mean()]
m = folium.Map(location=center, zoom_start=9, tiles='CartoDB positron')

for _, row in df_map.groupby(['latitude','longitude','label']).first().reset_index().iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color='white', weight=1,
        fill=True, fill_color=color_map.get(row['label'],'#888'),
        fill_opacity=0.85,
        tooltip=f"{row['district']} — {row['label']}",
        popup=folium.Popup(
            f"<b>{row['district']} ต.{row['sub-district']}</b><br>กลุ่ม: {row['label']}",
            max_width=250)
    ).add_to(m)

legend_html = '<div style="position:fixed;bottom:40px;left:40px;z-index:1000;background:white;padding:12px 16px;border-radius:8px;box-shadow:2px 2px 8px rgba(0,0,0,.3);font-family:sans-serif;font-size:13px;line-height:2;">'
legend_html += '<b>Cluster</b><br>'
for label, color in color_map.items():
    legend_html += f'<span style="display:inline-block;width:14px;height:14px;background:{color};border-radius:50%;margin-right:6px;vertical-align:middle;"></span>{label}<br>'
legend_html += '</div>'
m.get_root().html.add_child(folium.Element(legend_html))

m


จำนวนหน่วยต่อ cluster:
label
C2: Low Engagement (ประชาชน)            288
C0: Low Engagement (กล้าธรรม)           128
C1: Low Engagement (ประชาชน)            122
C3: Low Engagement (รวมไทยสร้างชาติ)     92

ค่าเฉลี่ยรายกลุ่ม:
                                      share_ประชาชน  share_เพื่อไทย  share_กล้าธรรม  share_ภูมิใจไทย  share_รวมไทยสร้างชาติ  turnout  invalid_rate
label                                                                                                                                             
C0: Low Engagement (กล้าธรรม)                 0.245           0.155           0.213            0.049                  0.058    0.654         0.045
C1: Low Engagement (ประชาชน)                  0.276           0.233           0.049            0.063                  0.064    0.566         0.075
C2: Low Engagement (ประชาชน)                  0.331           0.229           0.053            0.073                  0.052    0.653         0.042
C3: Low Engagement (รวมไทยสร้างชาติ)    